# STEP 5 : เว็บ (หลังบ้าน + หน้าบ้าน) และ Deploy
**ไฟล์คู่กัน:** `web/backend/app.py`, `web/frontend/*`, `06_export_onnx.py`, `05_deploy_huggingface.py`

```
              รันในเครื่อง (localhost)                        ออนไลน์ (Hugging Face, ฟรี)
 ┌──────────────┐  POST /api/predict  ┌──────────────┐      ┌──────────────────────────────┐
 │ หน้าบ้าน      │ ──────────────────▶ │ หลังบ้าน       │      │ หน้าบ้าน + โมเดล .onnx           │
 │ index.html   │ ◀────── JSON ─────── │ FastAPI+YOLO │      │ รันโมเดลในเบราว์เซอร์ (infer.js)   │
 └──────────────┘                     └──────────────┘      └──────────────────────────────┘
```
หน้าเว็บเลือกเองอัตโนมัติ: ถ้าเจอหลังบ้านจะใช้ API ถ้าไม่เจอจะรันในเบราว์เซอร์

In [1]:
# ===== ตั้งค่าเริ่มต้น (ทุก notebook ใช้เหมือนกัน) =====
import os, sys, json
from pathlib import Path
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # กัน error OMP บน Windows

# notebook อยู่ใน final_dip/scripts/  ->  ย้ายไปทำงานที่ final_dip/ (path ทุกอย่างอ้างอิงจากที่นี่)
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
os.chdir(ROOT)
CLASSES = ["airplane", "boat", "car", "motorcycle"]
print("ทำงานที่:", ROOT)
import time, subprocess, requests
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt

ทำงานที่: C:\code\ImageProcessing\yolo\yolo_lab\final_dip


## 5.1 หลังบ้าน (FastAPI) : ลองเรียก API
`TestClient` คือการจำลองเรียก API โดยไม่ต้องเปิด server จริง

In [2]:
sys.path.insert(0, str(ROOT))
from fastapi.testclient import TestClient
from web.backend.app import app

client = TestClient(app)
print("GET /api/health ->", client.get("/api/health").json())

[backend] loaded rect: C:\code\ImageProcessing\yolo\yolo_lab\final_dip\weights\rect_best.pt
[backend] loaded poly: C:\code\ImageProcessing\yolo\yolo_lab\final_dip\weights\poly_best.pt


GET /api/health -> {'status': 'ok', 'models': ['rect', 'poly'], 'classes': ['airplane', 'boat', 'car', 'motorcycle']}


In [3]:
with open("web/frontend/samples/airplane_008.jpg", "rb") as f:
    res = client.post("/api/predict", files={"file": f}, data={"mode": "poly", "conf": 0.25})
out = res.json()
for d in out["detections"]:
    d["polygon"] = d["polygon"][:3] + [f"... อีก {len(d['polygon']) - 3} จุด"]   # ตัดให้สั้นตอนแสดง
print("POST /api/predict  status =", res.status_code)
print(json.dumps(out, indent=2, ensure_ascii=False))

POST /api/predict  status = 200
{
  "mode": "poly",
  "image_width": 640,
  "image_height": 428,
  "inference_ms": 4080.1,
  "count": 1,
  "top_class": "airplane",
  "detections": [
    {
      "class_id": 0,
      "class_name": "airplane",
      "confidence": 0.9729,
      "box": [
        125.0,
        110.8,
        398.3,
        339.8
      ],
      "polygon": [
        [
          310.0,
          111.4
        ],
        [
          308.8,
          112.6
        ],
        [
          307.5,
          112.6
        ],
        "... อีก 397 จุด"
      ]
    }
  ]
}


## 5.2 เปิด server จริง แล้วลองบนเบราว์เซอร์
ตั้ง `START_SERVER = True` แล้วรัน → เปิด http://localhost:8000 (หน้าเอกสาร API อยู่ที่ http://localhost:8000/docs)

In [4]:
START_SERVER = False
if START_SERVER:
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "web.backend.app:app", "--port", "8000"])
    time.sleep(15)
    print(requests.get("http://localhost:8000/api/health").json())
    print("เปิด http://localhost:8000")
else:
    print("ข้าม (ตั้ง START_SERVER = True เพื่อเปิด server)")

ข้าม (ตั้ง START_SERVER = True เพื่อเปิด server)


In [5]:
# ปิด server (รันเมื่อใช้เสร็จ)
if START_SERVER:
    server.terminate()

## 5.3 แปลงโมเดลเป็น ONNX เพื่อให้เบราว์เซอร์รันได้
hosting ฟรีไม่มี Python → แปลงเป็นไฟล์ `.onnx` แล้วใช้ ONNX Runtime Web (JavaScript) รันแทน
แปลงไว้แล้ว (`EXPORT = False`) ถ้าเทรนใหม่ให้ตั้งเป็น `True`

In [6]:
EXPORT = False
if EXPORT:
    !{sys.executable} scripts/06_export_onnx.py

import onnxruntime as ort
for m in ["rect", "poly"]:
    p = Path(f"web/frontend/models/{m}.onnx")
    s = ort.InferenceSession(str(p))
    print(f"{p}  ({p.stat().st_size/1e6:.1f} MB)")
    print("   input :", [(i.name, i.shape) for i in s.get_inputs()])
    print("   output:", [(o.name, o.shape) for o in s.get_outputs()])

web\frontend\models\rect.onnx  (9.7 MB)
   input : [('images', [1, 3, 512, 512])]
   output: [('output0', [1, 8, 5376])]


web\frontend\models\poly.onnx  (11.0 MB)
   input : [('images', [1, 3, 512, 512])]
   output: [('output0', [1, 40, 5376]), ('output1', [1, 32, 128, 128])]


**อ่านค่า output**
- `output0 = [1, 8, 5376]` (rect) → 5376 กล่องที่เป็นไปได้ แต่ละกล่องมี 8 ค่า = `cx, cy, w, h` + คะแนน 4 คลาส
- `output0 = [1, 40, 5376]` (poly) → เพิ่มอีก 32 ค่า = สัมประสิทธิ์ของ mask
- `output1 = [1, 32, 128, 128]` (poly) → mask พื้นฐาน 32 แผ่น เอามาผสมตามสัมประสิทธิ์จะได้ mask ของวัตถุ

## 5.4 สิ่งที่ `infer.js` ทำในเบราว์เซอร์ (เขียนเป็น Python ให้ดู)
ขั้นตอนเดียวกับโค้ด JavaScript ทุกบรรทัด: **letterbox → รันโมเดล → กรอง conf → NMS → แปลงพิกัดกลับ**

In [7]:
IMGSZ = 512
sess = ort.InferenceSession("web/frontend/models/rect.onnx")
img = cv2.cvtColor(cv2.imread("web/frontend/samples/car_013.jpg"), cv2.COLOR_BGR2RGB)
h0, w0 = img.shape[:2]

# 1) letterbox : ย่อให้ด้านยาว = 512 แล้วเติมขอบเทา (114) ให้เป็นสี่เหลี่ยมจัตุรัส
scale = min(IMGSZ / w0, IMGSZ / h0)
nw, nh = round(w0 * scale), round(h0 * scale)
padX, padY = round((IMGSZ - nw) / 2 - 0.1), round((IMGSZ - nh) / 2 - 0.1)
canvas = np.full((IMGSZ, IMGSZ, 3), 114, np.uint8)
canvas[padY:padY + nh, padX:padX + nw] = cv2.resize(img, (nw, nh))

# 2) เป็น tensor [1, 3, 512, 512] ค่า 0-1
x = canvas.transpose(2, 0, 1)[None].astype(np.float32) / 255

# 3) รันโมเดล
out = sess.run(None, {"images": x})[0][0]          # [8, 5376]
boxes, scores = out[:4].T, out[4:].T               # [5376, 4], [5376, 4]
cls, conf = scores.argmax(1), scores.max(1)

# 4) กรอง conf >= 0.25
keep = conf >= 0.25
print("กล่องทั้งหมด:", len(conf), "-> หลังกรอง conf:", keep.sum())
boxes, cls, conf = boxes[keep], cls[keep], conf[keep]
xyxy = np.c_[boxes[:, 0] - boxes[:, 2] / 2, boxes[:, 1] - boxes[:, 3] / 2,
             boxes[:, 0] + boxes[:, 2] / 2, boxes[:, 1] + boxes[:, 3] / 2]

# 5) NMS : เรียงคะแนนมาก -> น้อย ตัดกล่องคลาสเดียวกันที่ซ้อนกัน (IoU > 0.7)
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)
final = []
for i in conf.argsort()[::-1]:
    if all(cls[j] != cls[i] or iou(xyxy[j], xyxy[i]) < 0.7 for j in final):
        final.append(i)
print("หลัง NMS:", len(final))

# 6) แปลงพิกัดกลับเป็นของภาพจริง (หักขอบเทา หารอัตราย่อ)
names = ["airplane", "boat", "car", "motorcycle"]
print("\nผลจาก ONNX (แบบที่เว็บทำ):")
for i in final:
    b = (xyxy[i] - [padX, padY, padX, padY]) / scale
    print(f"   {names[cls[i]]:<10} {conf[i]:.3f}  box = {b.round().astype(int).tolist()}")

from ultralytics import YOLO
# เทียบกับ ultralytics ที่รันไฟล์ .onnx เดียวกัน (ภาพเข้า 512x512 เหมือนกัน -> ผลต้องตรงกัน)
# หมายเหตุ: ถ้าใช้ไฟล์ .pt ultralytics จะเติมขอบแค่พอดี (เช่น 512x352) ผลจึงต่างกันเล็กน้อย
r = YOLO("web/frontend/models/rect.onnx", task="detect").predict("web/frontend/samples/car_013.jpg", imgsz=512, verbose=False)[0]
print("\nผลจาก ultralytics (ไฟล์ onnx เดียวกัน) เพื่อเทียบ:")
for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy):
    print(f"   {names[int(c)]:<10} {float(s):.3f}  box = {b.round().int().tolist()}")

กล่องทั้งหมด: 5376 -> หลังกรอง conf: 10
หลัง NMS: 2

ผลจาก ONNX (แบบที่เว็บทำ):
   car        0.856  box = [451, 198, 641, 358]
   car        0.788  box = [258, 234, 482, 324]
Loading web/frontend/models/rect.onnx for ONNX Runtime inference...


Using ONNX Runtime 1.31.0 with CPUExecutionProvider



ผลจาก ultralytics (ไฟล์ onnx เดียวกัน) เพื่อเทียบ:
   car        0.856  box = [451, 198, 640, 359]
   car        0.788  box = [258, 234, 482, 324]


## 5.5 Deploy ขึ้น Hugging Face Spaces
ใช้ Space แบบ **static** (ฟรี) อัปโหลดทุกไฟล์ใน `web/frontend/` รวมไฟล์ `.onnx`
ต้อง login ก่อนด้วย `hf auth login` (ทำครั้งเดียว) · deploy ไว้แล้ว (`DEPLOY = False`)

In [8]:
DEPLOY = False
SPACE = "witthaya22/final-dip"
if DEPLOY:
    !{sys.executable} scripts/05_deploy_huggingface.py --space {SPACE}
else:
    print("ข้าม (deploy ไว้แล้ว)")

ข้าม (deploy ไว้แล้ว)


In [9]:
# เช็คว่าเว็บออนไลน์ และไฟล์ครบ
URL = "https://witthaya22-final-dip.static.hf.space"
rows = []
for f in ["", "app.js", "infer.js", "style.css", "models/rect.onnx", "models/poly.onnx", "samples/samples.json"]:
    r = requests.get(f"{URL}/{f}", timeout=60)
    rows.append({"file": "/" + f, "status": r.status_code, "size": f"{len(r.content)/1e3:,.0f} KB"})
print("เว็บ:", URL)
pd.DataFrame(rows)

เว็บ: https://witthaya22-final-dip.static.hf.space


,file,status,size
0,/,200,5 KB
1,/app.js,200,14 KB
2,/infer.js,200,9 KB
3,/style.css,200,5 KB
4,/models/rect.onnx,200,"9,735 KB"
5,/models/poly.onnx,200,"11,009 KB"
6,/samples/samples.json,200,0 KB
